# SSM Centric Indexer
```
ssm{}
    |____ consequence[]
    |           |_____ transcript{}
    |                        |_____ gene{}
    |                        |_____ annotation{}
    |____ occurrence[]
                |_____ case{}
                         |____ observation[]
```

In [1]:
import os
import requests
import uuid
%load_ext autoreload
from exports.mappings import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

## Load combined maf into spark

In [3]:
url = 's3a://test/KIRC_KIRP_KICH_mafs.csv'
    
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true')\
                .load(url)\
                .drop_duplicates()

## Rename and select desired columns in the mafs

In [4]:
%autoreload
from exports.utils import (
    maf_annotation_map,
    maf_gene_map,
    maf_observation_map,
    maf_ssm_map,
    maf_transcript_map,
    tumor_genotype_map,
    tumor_validation_map,
    normal_genotype_map,
    sample_map,
    input_bam_map,
    read_depth_map,
    maf_cols
)

maf_df = df.select(*( col(v).alias(k) for k,v in maf_cols.items() ))

## Augment maf df by extracting submitter_id and creating ssm_uuids

In [5]:
maf_df = maf_df.withColumn('_case_submitter_id',
                           regexp_extract(col('tumor_sample_barcode'),
                                          '([A-Z]{4}-[A-Z0-9]{2}-[A-Z0-9]{4})',1))
maf_ssm_map.update({'_case_submitter_id':'_case_submitter_id'})

In [6]:
def ssm_uuid(chromosome, start_position, ref_allele, tumor_allele):
    '''
    SNP: "{chromosome}:g.{start_position}{reference_allele}>{tumor_allele}"
    DEL: "{chromosome}:g.{start_position}del{reference_allele}"
    INS: "{chromosome}:g.{start_position}_{end_position}ins{tumor_allele}"
    '''
    chromosome = chromosome.replace('chr','')
    label = '{}:g.{}:{}>{}'.format(chromosome, start_position, ref_allele, tumor_allele)
    return str(uuid.uuid5(uuid.UUID('d15296a3-38ed-412e-8ace-75e235f82f55'), label))

ssm_uuid_udf =udf(ssm_uuid, StringType())
maf_df = maf_df.withColumn('ssm_id', ssm_uuid_udf(col('chromosome'), col('start_position'), col('reference_allele'), col('tumor_allele')))
maf_observation_map.update({'ssm_id':'ssm_id'})
maf_ssm_map.update({'ssm_id':'ssm_id'})

## Slice and dice until we get to the format we want

### SSM df

In [7]:
ssm_df = maf_df.select(*( col(k) for k in maf_ssm_map.keys() ))\
               .drop_duplicates()

## Transcript + Annotation + Gene
```
consequence[]
    |_____ transcript{}
                |_____ gene{}
                |_____ annotation{}
```

In [8]:
# Select annotation and gene into nested format
tran_df = maf_df.select('gene_id', struct(
                                struct(*maf_annotation_map.keys()).alias('annotation'),
                                struct(*maf_gene_map.keys()).alias('gene'),
                                *set(maf_transcript_map.keys()) - set(['gene_id'])
                              ).alias('transcript'))\
                            .drop_duplicates()

In [9]:
consequence_df = tran_df.select('gene_id', struct('transcript').alias('consequence'))\
                        .groupBy('gene_id')\
                        .agg(collect_list('consequence').alias('consequence'))

## Observation + Case + Occurance
```
occurrence[]
    |_____ case{}
             |____ observation[]
```

### Observation df

In [10]:
observation_df = maf_df.select('_case_submitter_id',
                               *(maf_observation_map.keys()
                                 +normal_genotype_map.keys()
                                 +tumor_genotype_map.keys()
                                 +tumor_validation_map.keys()
                                 +read_depth_map.keys()
                                 +input_bam_map.keys()
                                 +sample_map.keys()))

observation_df = observation_df.select('_case_submitter_id',
                                       struct(*normal_genotype_map.keys()).alias('normal_genotype'),
                                       struct(*tumor_genotype_map.keys()).alias('tumor_genotype'),
                                       struct(*tumor_validation_map.keys()).alias('validation'),
                                       struct(*read_depth_map.keys()).alias('read_depth'),
                                       struct(*input_bam_map.keys()).alias('input_bam_file'),
                                       struct(*sample_map.keys()).alias('sample'),
                                       *maf_observation_map.keys())\
                                .drop('gene_id')\
                                .drop_duplicates()

### Get case dataframe from existing graph

In [21]:
#doc = requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph_35/_search').json()['hits']['hits'][0]['_source']
case_df = sqlContext.read.format("es")\
    .option('es.nodes', 'elasticsearch.service.consul')\
    .option('es.read.field.include', 'case_id,submitter_id,state,project.*,program.*,exposures.*,demographic.*,*_ids,summary.*)')\
    .option('es.read.field.as.array.include','')\
    .option('es.resource.read', 'gdc_from_graph/case')\
    .option('es.nodes.resolve.hostname','false')\
    .load("gdc_from_graph")

In [12]:
# Merge observation with Case
occurrence_df = case_df.join(observation_df, case_df.submitter_id == observation_df._case_submitter_id, 'left')\
                        .select('ssm_id', struct(struct(struct(observation_df.drop('_case_submitter_id').drop('ssm_id').columns).alias('observation'),*case_df.columns).alias('case')).alias('occurrence'))\
                        .groupby('ssm_id')\
                        .agg(collect_list('occurrence').alias('occurrence'))

In [13]:
#occurrence_df.printSchema()

## Assemble constituent parts
```
ssm{}
    |____ consequence[]
    |           |_____ transcript{}
    |                        |_____ gene{}
    |                        |_____ annotation{}
    |____ occurrence[]
                |_____ case{}
                         |____ observation[]
```

In [14]:
ssm_centric = ssm_df.join(occurrence_df, ssm_df.ssm_id == occurrence_df.ssm_id, 'inner')\
                    .join(consequence_df, ssm_df.gene_id == consequence_df.gene_id, 'inner')\
                    .drop('_case_submitter_id')\
                    .drop(consequence_df.gene_id)\
                    .drop('gene_id')\
                    .drop('submitter_id')

## Export df to es

In [15]:
sqlContext.sql("set spark.sql.shuffle.partitions=2048")

DataFrame[key: string, value: string]

In [16]:
index = 'dan-r1-ssm'

In [17]:
%autoreload
from exports.mappings import CaseMapper, SSMMapper
m = SSMMapper()

In [18]:
m.mapping['_size'] =  {"enabled": 'true'}
#m.mapping['dynamic'] = 'true'
#m.mapping['properties']['ssm']['dynamic'] = 'true'
#m.mapping['properties']['gene']['properties']['ssm']['dynamic'] = 'true'
#m.mapping['properties']['gene']['properties']['ssm']['properties']['observation']['dynamic'] = 'true'
#m.mapping['properties']['case']['properties']['files']['properties']['cases']['dynamic'] = 'true'

In [19]:
import json

print requests.delete('http://elasticsearchvis.service.consul:9200/{}'.format(index)).json()

data = json.dumps({"settings":{"index":{
                "refresh_interval":"10m",
                "number_of_shards":20,
                "number_of_replicas":1,
                "mapper.dynamic":False,
                "mapping.nested_fields.limit":100,
                "mapping.total_fields.limit":2000
            }},"mappings":{
                "ssm":m.mapping
            }})
#print requests.put('http://localhost:9200/test/', data=data).json()
print requests.put('http://elasticsearchvis.service.consul:9200/{}'.format(index), data=data).json()

{u'acknowledged': True}
{u'acknowledged': True, u'shards_acknowledged': True}


In [20]:
#%%time
ssm_centric.write.format('org.elasticsearch.spark.sql')\
                    .option('es.nodes', 'elasticsearchvis.service.consul')\
                    .option('es.nodes.resolve.hostname','false')\
                    .option('es.resource.write', '{}/ssm'.format(index))\
                    .option('es.http.timeout', '1m')\
                    .option('es.http.retries', '300')\
                    .option('es.batch.write.retry.count', '100')\
                    .option('es.batch.write.retry.wait', '1m')\
                    .option('es.batch.size.bytes','10mb')\
                    .option('es.batch.size.entries', '5000')\
                    .option('es.mapping.id','ssm_id')\
                    .save('{}/ssm'.format(index))
            
#requests.post('http://localhost:9200/test-case/_refresh')